We would be needing a compute in order to run these services  
As of now Absent in serverless services

In [0]:
from pyspark.sql import functions as f 
from pyspark.sql.functions import * 


In [0]:
df = (spark.readStream.format('cloudfiles')
      .option('cloudfiles.format', 'csv')
      .option('cloudfiles.schemaLocation', '/Volumes/namaste_catalog/vineetdb/testvolume/schemalocation/')
      .option('cloudfiles.inferSchema',True)
      .option('cloudfiles.header',True)
      .option("cloudFiles.validateOptions", "false")
      .load("/Volumes/namaste_catalog/vineetdb/testvolume/files/")
      .withColumn('file_name', f.col('_metadata.file_name') )
      .withColumn('ingest_ts', current_timestamp())
      
         )

In [0]:
newcols = [col.replace(' ','_').lower() for col in df.columns]
print(newcols)
newdf = df.toDF(*newcols)


### available=Now and  .trigger(once=True)\ are for one time use and they can run on serverless compute as well
- We need to make a separate cluster for 
- processingTime = '10 seconds ' 
- and continuous 


![image_1779633517276.png](./image_1779633517276.png "image_1779633517276.png")

In [0]:
newdf.writeStream \
  .format("delta") \
  .outputMode("append") \
  .trigger(once=True)\
  .option("checkpointLocation", "/Volumes/namaste_catalog/vineetdb/testvolume/checkpoint/") \
  .table("namaste_catalog.vineetdb.str_tbl")


In [0]:
%sql 
select file_name, count(1)  as total_records
from namaste_catalog.vineetdb.str_tbl
group by file_name

### Different type of output mode Append Complete Update